In [ ]:
# =============================================================================
# 🏆 FINAL MRI BRAIN TUMOR PROJECT (COMPLETE PIPELINE)
# Model: Xception | Accuracy Goal: >95% | Viz: Jet/Thermal Spectrum
# =============================================================================

import os
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# -----------------------------------------------------------------------------
# 1️⃣ CONFIGURATION & DIRECTORIES
# -----------------------------------------------------------------------------
print("⚡ Pipeline Starting...")

TRAIN_DIR = "/kaggle/input/dataset1/Training"
TEST_DIR  = "/kaggle/input/dataset1/Testing"
SINGLE_IMG_PATH = "/kaggle/input/myimage2/7 no.jpg" # Image for Thermal Viz

OUTPUT_DIR = "/kaggle/working/Project_Results"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

IMG_SIZE = (299, 299) # Best for Xception
BATCH_SIZE = 16
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]

# -----------------------------------------------------------------------------
# 2️⃣ PREPROCESSING (CLAHE - Medical Standard)
# -----------------------------------------------------------------------------
def preprocess_mri(img):
    img = img.astype("uint8")
    img = cv2.resize(img, IMG_SIZE)
    # Apply CLAHE to L-channel
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl,a,b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
    img = img.astype("float32") / 255.0
    return img

# -----------------------------------------------------------------------------
# 3️⃣ DATA GENERATORS (Split Strategy for High Accuracy)
# -----------------------------------------------------------------------------
# We split 'Training' folder 80/20. We use 'Testing' folder ONLY for final report.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_mri,
    validation_split=0.2, # Internal split
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.2, horizontal_flip=True, fill_mode='nearest'
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", shuffle=True
)

val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", shuffle=False
)

# -----------------------------------------------------------------------------
# 4️⃣ MODEL: XCEPTION (SOTA for Texture)
# -----------------------------------------------------------------------------
base_model = Xception(weights="imagenet", include_top=False, input_shape=(299, 299, 3))
base_model.trainable = True # Unfreeze for fine-tuning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x) 
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adamax(learning_rate=0.0001), loss="categorical_crossentropy", metrics=["accuracy"])

# -----------------------------------------------------------------------------
# 5️⃣ TRAINING
# -----------------------------------------------------------------------------
print("🚀 Starting Training...")
callbacks = [
    ModelCheckpoint(os.path.join(OUTPUT_DIR, "best_model.keras"), monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1)
]

history = model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=callbacks)

# -----------------------------------------------------------------------------
# 6️⃣ EVALUATION: PLOTS & CONFUSION MATRIX
# -----------------------------------------------------------------------------
print("📊 Generating Report...")

# A. Training Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"))
plt.show()

# B. Confusion Matrix on External Test Set
test_datagen_eval = ImageDataGenerator(preprocessing_function=preprocess_mri)
test_gen_final = test_datagen_eval.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", shuffle=False
)

y_pred = np.argmax(model.predict(test_gen_final), axis=1)
y_true = test_gen_final.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Confusion Matrix")
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"))
plt.show()

# C. Classification Report
report = classification_report(y_true, y_pred, target_names=CLASSES)
print(report)
with open(os.path.join(OUTPUT_DIR, "final_metrics.txt"), "w") as f:
    f.write(report)

# -----------------------------------------------------------------------------
# 7️⃣ VISUALIZATION: JET/THERMAL SPECTRUM (The Exact Look)
# -----------------------------------------------------------------------------
def visualize_jet_spectrum(img_path):
    if not os.path.exists(img_path): return

    # Load & Preprocess
    img_bgr = cv2.imread(img_path)
    img_resized = cv2.resize(img_bgr, IMG_SIZE)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    
    tensor_img = preprocess_mri(img_bgr)
    tensor_img = np.expand_dims(tensor_img, axis=0)

    # Predict
    preds = model.predict(tensor_img)
    idx = np.argmax(preds[0])
    label = CLASSES[idx]
    conf = preds[0][idx]

    # Grad-CAM
    last_conv_layer = "block14_sepconv2_act" # Xception specific
    grad_model = Model([model.inputs], [model.get_layer(last_conv_layer).output, model.output])
    
    with tf.GradientTape() as tape:
        conv_out, preds_vec = grad_model(tensor_img)
        loss = preds_vec[:, idx]
        
    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_out[0]), axis=-1)
    
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) != 0: heatmap /= np.max(heatmap)
    heatmap = heatmap.numpy()
    heatmap_resized = cv2.resize(heatmap, (IMG_SIZE[1], IMG_SIZE[0]))

    # 🎨 MIXING: 40% Structure + 60% Heatmap -> JET Colormap
    gray_norm = img_gray.astype("float32") / 255.0
    combined = (gray_norm * 0.4) + (heatmap_resized * 0.6)
    combined_uint8 = np.uint8(255 * combined)
    final_jet_image = cv2.applyColorMap(combined_uint8, cv2.COLORMAP_JET)

    # Background Masking
    _, mask = cv2.threshold(img_gray, 10, 255, cv2.THRESH_BINARY)
    final_jet_image = cv2.bitwise_and(final_jet_image, final_jet_image, mask=mask)

    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(final_jet_image, cv2.COLOR_BGR2RGB))
    plt.title(f"Thermal Scan: {label} ({conf:.1%})")
    plt.axis("off")
    plt.savefig(os.path.join(OUTPUT_DIR, "thermal_output.png"))
    plt.show()

visualize_jet_spectrum(SINGLE_IMG_PATH)
print(f"\n✅ Project Complete! Results saved in: {OUTPUT_DIR}")

2026-03-16 04:41:48.304965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773636108.511276      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773636108.570516      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773636109.078563      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773636109.078607      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773636109.078610      55 computation_placer.cc:177] computation placer alr

⚡ Pipeline Starting...
Found 4571 images belonging to 4 classes.
Found 1141 images belonging to 4 classes.


I0000 00:00:1773636123.741991      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773636123.747794      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
🚀 Starting Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


...

 15/286 ━━━━━━━━━━━━━━━━━━━━ 2:02 453ms/step - accuracy: 0.2628 - loss: 2.8576

2026-03-16 04:43:37.921639: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:43:38.145151: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:43:38.838870: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:43:39.035834: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:43:39.782795: E external/local_xla/xla/stream_

286/286 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - accuracy: 0.5723 - loss: 1.4231

2026-03-16 04:47:10.739057: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:47:10.973296: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:47:11.642899: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:47:11.810110: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-16 04:47:12.367658: E external/local_xla/xla/stream_


Epoch 1: val_accuracy improved from -inf to 0.60736, saving model to /kaggle/working/Project_Results/best_model.keras
286/286 ━━━━━━━━━━━━━━━━━━━━ 319s 854ms/step - accuracy: 0.5728 - loss: 1.4214 - val_accuracy: 0.6074 - val_loss: 0.9257 - learning_rate: 1.0000e-04
Epoch 2/20
286/286 ━━━━━━━━━━━━━━━━━━━━ 0s 463ms/step - accuracy: 0.8515 - loss: 0.4709
Epoch 2: val_accuracy improved from 0.60736 to 0.72656, saving model to /kaggle/working/Project_Results/best_model.keras
286/286 ━━━━━━━━━━━━━━━━━━━━ 159s 556ms/step - accuracy: 0.8515 - loss: 0.4709 - val_accuracy: 0.7266 - val_loss: 0.6824 - learning_rate: 1.0000e-04
Epoch 3/20
286/286 ━━━━━━━━━━━━━━━━━━━━ 0s 466ms/step - accuracy: 0.8773 - loss: 0.3706
Epoch 3: val_accuracy improved from 0.72656 to 0.86941, saving model to /kaggle/working/Project_Results/best_model.keras
286/286 ━━━━━━━━━━━━━━━━━━━━ 160s 558ms/step - accuracy: 0.8773 - loss: 0.3705 - val_accuracy: 0.8694 - val_loss: 0.3803 - learning_rate: 1.0000e-04
Epoch 4/20
286/2